In [1]:
from nsnet2_denoiser import NSnet2Enhancer

from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore, ScaleInvariantSignalDistortionRatio
from torchaudio.transforms import Resample

import torch
import torchaudio
import numpy as np

import os

In [2]:
import random

SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

In [3]:
CHKP_DIR = "checkpoints"

In [4]:
from src.fspen_configs import *
from models.fspen import *

from src.fspen_configs import *

configs = TrainConfig_48kHz_enc_ext() # TrainConfig_explicit_unfold()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension_ver2_abs_pha(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48kHz_enc_ext_1986#0.pt"), map_location="cpu",  weights_only=False)
fspen.load_state_dict(state_d["model_state_dict"])
fspen.eval()

FullSubPathExtension_ver2_abs_pha(
  (full_band_encoder): FullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
  )
  (sub_band_encoder): SubBandEncoder_ver2(
    (sub_band_encoders): M

In [5]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [6]:
SR = 48_000

# CLEAN_PATH = "data/DS_10283_2791/clean_testset_wav/"
# NOISE_PATH = "data/DS_10283_2791/noisy_testset_wav/"

CLEAN_PATH = "data/DS_10283_2791/clean_testset_wav"
NOISE_PATH = "data/DS_10283_2791/noisy_testset_wav"

def list_files_walk(root: str):
    res = []
    for dirpath, dirnames, filenames in os.walk(root, topdown=True):  # рекурсивно обходим дерево [web:4][web:17]
        dirnames.sort()   # фиксируем порядок обхода поддиректорий [web:23]
        filenames.sort()  # фиксируем порядок файлов в каталоге
        for name in filenames:
            res.append(os.path.join(dirpath, name))
    return res

# clean_paths = [os.path.join(CLEAN_PATH, x) for x in os.listdir(CLEAN_PATH)]# [:100]
# noise_paths = [os.path.join(NOISE_PATH, x) for x in os.listdir(NOISE_PATH)]# [:100]
noise_paths = list_files_walk(NOISE_PATH)
clean_paths = list_files_walk(CLEAN_PATH)

test_data = list(zip(noise_paths, clean_paths))# [:10]

BATCH_SIZE = 32
DEVICE = "cuda:0"

In [7]:
srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(16_000, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)
sisdr = ScaleInvariantSignalDistortionRatio().to
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [8]:
N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate

In [9]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [10]:
window = vorbis_window(N_FFTS).to(DEVICE)
fspen = fspen.to(DEVICE)

In [11]:
noise_paths[0].split('/')[-1][:-4]

'p232_001'

In [12]:
OUTPUT_PATH = "data/voicebank_enhanced/"

In [13]:
# from src.dataset import *

# clean_dataset = TRUNetDataset(CLEAN_PATH, sr=SR, noise_dir=None, rir_dir=None, snr=[0, 5, 10, 15], rir_proba=1.0, noise_proba=1.0, rir_target=False, return_noise=False, return_rir=False)
# noise_dataset = TRUNetDataset(NOISE_PATH, sr=SR, noise_dir=None, rir_dir=None, snr=[0, 5, 10, 15], rir_proba=1.0, noise_proba=1.0, rir_target=False, return_noise=False, return_rir=False)

In [14]:
from src.utils import model_eval, model_eval_old
from tqdm import tqdm
from scipy.io.wavfile import write
from src.dataset import *

# write(AUDIO_PATH[:-4] + "_unfold.wav", SR, out_wave.cpu().detach().numpy())

def get_metrics(data, device="cpu"):
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    dnsmos_scores = []

    nisqa_scores_input = []
    pesq_scores_input = []
    stoi_scores_input = []
    srmr_scores_input = []
    dnsmos_scores_input = []

    nisqa_scores_target = []
    pesq_scores_target = []
    stoi_scores_target = []
    srmr_scores_target = []
    dnsmos_scores_target = []

    with torch.no_grad():
        for input_path, target_path in tqdm(test_data):
        # for i in range(len(clean_dataset)):
            # signal, _, _, _= noise_dataset[i]
            # target, _, _, _ = clean_dataset[i] 
            
            signal, signal_sr = torchaudio.load(input_path)
            target, target_sr = torchaudio.load(target_path)

            signal = signal.to(device)
            target = target.to(device)
            
            input_signal, _ = SignalDataset.normalize_audio(signal)
            # target, signal = SignalDataset.normalize_audio(target, signal)
            # target, _ = SignalDataset.normalize_audio(target)
            # resampler = Resample(signal_sr, SR)
            # sig_resampled = resampler(signal).to(device)
            window = vorbis_window(N_FFTS).to(device)
            spec = torch.stft(
                input_signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            ) 

            output, _ = model_eval(fspen, spec, configs, device, hid_size=HID_SIZE)

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                       window=window,
                       # onesided=True,
                       return_complex=False,
                       normalized=True,
                       center=True)

            output = output / (output.abs().max() / signal.abs().max())
            
            output = output.reshape(-1)

            # write(OUTPUT_PATH + input_path.split('/')[-1][:-4] + "_enhanced.wav", signal_sr, output.cpu().detach().numpy())
            # write(OUTPUT_PATH + str(i) + "_enhanced.wav", SR, output[0].cpu().detach().numpy())
            output = output.unsqueeze(0)
            min_l = min(output.shape[-1], target.shape[-1])

            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            # nisqa_score_input, _, _ = process(signal.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            # nisqa_score_target, _, _ = process(target.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            output = output[:, :min_l]
            signal = signal[:, :min_l]
            target = target[:, :min_l]

            # stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            # signal = signal.to(device)
            # stoi_score_input = stoi(signal[..., :min_l], target[..., :min_l])
            # srmr_score = srmr(output.detach().cpu())
            
            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target = resampler(target.cpu()).cuda()
            signal = resampler(signal.cpu()).cuda()

            stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            signal = signal.to(device)
            # stoi_score_input = stoi(signal[..., :min_l], target[..., :min_l])
            
            dnsmos_score = dnsmos(output.detach())
            # dnsmos_score_input = dnsmos(signal.detach())
            # dnsmos_score_target = dnsmos(target.detach())
            
            # min_l = min(output.shape[-1], target.shape[-1])

            srmr_score = srmr(output.detach().cpu())
            # srmr_score_input = srmr(signal.detach().cpu())
            # srmr_score_target = srmr(target.detach().cpu())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
                # pesq_score_input = pesq(signal[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dnsmos_scores.append(dnsmos_score.cpu())

            # nisqa_scores_input.append(nisqa_score_input[0])
            # srmr_scores_input.append(srmr_score_input)
            # stoi_scores_input.append(stoi_score_input.cpu())
            # pesq_scores_input.append(pesq_score_input.cpu())
            # dnsmos_scores_input.append(dnsmos_score_input.cpu())

            # nisqa_scores_target.append(nisqa_score_target[0])
            # srmr_scores_target.append(srmr_score_target)
            # stoi_scores_target.append(1.0)
            # pesq_scores_target.append(1.0)
            # dnsmos_scores_target.append(dnsmos_score_target.cpu())

    nisqa_scores = torch.vstack(nisqa_scores).mean(dim=0)
    stoi_scores = torch.vstack(stoi_scores).mean(dim=0)
    srmr_scores = torch.vstack(srmr_scores).mean(dim=0)
    pesq_scores = torch.vstack(pesq_scores).mean(dim=0)
    dnsmos_scores = torch.vstack(dnsmos_scores).mean(dim=0)

    # nisqa_scores_input = torch.vstack(nisqa_scores_input).mean(dim=0)
    # stoi_scores_input = torch.vstack(stoi_scores_input).mean(dim=0)
    # srmr_scores_input = torch.vstack(srmr_scores_input).mean(dim=0)
    # pesq_scores_input = torch.vstack(pesq_scores_input).mean(dim=0)
    # dnsmos_scores_input = torch.vstack(dnsmos_scores_input).mean(dim=0)

    # nisqa_scores_target = torch.vstack(nisqa_scores_target).mean(dim=0)
    # stoi_scores_target = 1.0 # torch.vstack(stoi_scores_target).mean(dim=0)
    # srmr_scores_target = torch.vstack(srmr_scores_target).mean(dim=0)
    # pesq_scores_target = 4.5# torch.vstack(pesq_scores_target).mean(dim=0)
    # dnsmos_scores_target = torch.vstack(dnsmos_scores_target).mean(dim=0)

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dnsmos_scores}
    result_input = {"nisqa": nisqa_scores_input, "stoi": stoi_scores_input, "srmr": srmr_scores_input, "pesq": pesq_scores_input, "dnsmos": dnsmos_scores_input}
    result_target = {"nisqa": nisqa_scores_target, "stoi": stoi_scores_target, "srmr": srmr_scores_target, "pesq": pesq_scores_target, "dnsmos": dnsmos_scores_target}

    return result, result_input, result_target

In [15]:
metrics, metrics_input, metrics_target = get_metrics(test_data, device=DEVICE)

100%|██████████| 824/824 [12:21<00:00,  1.11it/s]


In [16]:
print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics["nisqa"])
print(f"STOI score: {-metrics['stoi']}")
print(f"SRMR score: {metrics['srmr']}")
print(f"PESQ-WB score: {metrics['pesq']}")
print(f"DNSMOS score: {metrics['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([3.8964, 4.0757, 3.8910, 3.9292, 3.9224])
STOI score: tensor([0.9060])
SRMR score: tensor([8.8912])
PESQ-WB score: tensor([2.6285])
DNSMOS score: tensor([3.3753, 3.3872, 3.9182, 3.0567], dtype=torch.float64)


In [17]:
metrics

{'nisqa': tensor([3.8964, 4.0757, 3.8910, 3.9292, 3.9224]),
 'stoi': tensor([-0.9060]),
 'srmr': tensor([8.8912]),
 'pesq': tensor([2.6285]),
 'dnsmos': tensor([3.3753, 3.3872, 3.9182, 3.0567], dtype=torch.float64)}

In [18]:
metrics = {k: [v, ] for k, v in metrics.items()}

df = pd.DataFrame(metrics)
df.to_csv("fspen_48khz_enc_ext_1986_voicebank.csv", index=False)

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([3.636, 3.858, 3.708, 3.782, 3.773])
STOI score: tensor([0.897])
SRMR score: tensor([8.201])
PESQ-WB score: tensor([2.230])
DNSMOS score: tensor([3.242], dtype=torch.float64)

In [19]:
# print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics_input["nisqa"])
# print(f"STOI score: {-metrics_input['stoi']}")
# print(f"SRMR score: {metrics_input['srmr']}")
# print(f"PESQ-WB score: {metrics_input['pesq']}")
# print(f"DNSMOS score: {metrics_input['dnsmos']}")

DNS
NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([2.3900, 2.6258, 3.6104, 3.0987, 2.6130])
STOI score: tensor([1.0000])
SRMR score: tensor([4.5781])
PESQ-WB score: tensor([4.6439])
DNSMOS score: tensor([3.0723, 3.0899, 2.6856, 2.3037], dtype=torch.float64)

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([3.0895, 2.7641, 3.5948, 3.5909, 3.4799])
STOI score: tensor([0.8855])
SRMR score: tensor([7.9794])
PESQ-WB score: tensor([1.9726])
DNSMOS score: tensor([3.0792, 3.3152, 3.0964, 2.6730], dtype=torch.float64)

In [20]:
# print("NISQA score (MOS, NOI, DISC, COL, LOUD):", metrics_target["nisqa"])
# print(f"STOI score: {metrics_target['stoi']}")
# print(f"SRMR score: {metrics_target['srmr']}")
# print(f"PESQ-WB score: {metrics_target['pesq']}")
# print(f"DNSMOS score: {metrics_target['dnsmos']}")

NISQA score (MOS, NOI, DISC, COL, LOUD): tensor([4.0741, 4.2455, 4.0119, 4.1209, 4.0925])
STOI score: 1.0
SRMR score: tensor([8.9431])
PESQ-WB score: 4.5
DNSMOS score: tensor([3.5542, 3.5086, 4.0323, 3.2156], dtype=torch.float64)